In [1]:
pip install pyodbc sqlalchemy pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
from sqlalchemy import create_engine
import pandas as pd
import numpy as np
import os
import pyodbc


In [3]:
SERVER = 'DESKTOP-6EJAH5F'          
DATABASE = 'Hospital_Project'
USERNAME = 'sa'              
PASSWORD = '1919' 

# SQLAlchemy engine for SQL Server
engine = create_engine(
    f"mssql+pyodbc://{USERNAME}:{PASSWORD}@{SERVER}/{DATABASE}"
    f"?driver=ODBC+Driver+17+for+SQL+Server"
)

# Path to the dataset files
DATA_PATH = "C:/Users/DELL/Documents/NareshIT/Data Analytics Screening Test Batch/Project_1/Data_Files/"
# pyodbc connection config
conn_str = (
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={SERVER};"
    f"DATABASE={DATABASE};"
    f"UID={USERNAME};"
    f"PWD={PASSWORD};"
)

In [4]:
# Connect and insert
try:
    conn = pyodbc.connect(conn_str)
    cursor = conn.cursor()
    print("SQL Server connection established successfully.")

    csv_files = [
        'admissions_raw.csv',
        'billing_claims_raw.csv',
        'departments.csv',
        'doctors.csv',
        'followups_raw.csv',
        'hospitals.csv',
        'lab_results_raw.csv',
        'patients_raw.csv',
        'pharmacy_orders_raw.csv',
        'treatments_raw.csv'
    ]

    for file in csv_files:
        df = pd.read_csv(os.path.join(DATA_PATH, file))
        table_name = file.replace('.csv', '').replace('_raw', '')
        df.to_sql(name=table_name, con=engine, if_exists='replace', index=False, schema='dbo')
        print(f"Table '{table_name}' created successfully.")

except pyodbc.Error as err:
    print(f"Error: {err}")

finally:
    if conn:
        cursor.close()
        conn.close()
        print("SQL Server connection closed.")

SQL Server connection established successfully.
Table 'admissions' created successfully.
Table 'billing_claims' created successfully.
Table 'departments' created successfully.
Table 'doctors' created successfully.
Table 'followups' created successfully.
Table 'hospitals' created successfully.
Table 'lab_results' created successfully.
Table 'patients' created successfully.
Table 'pharmacy_orders' created successfully.
Table 'treatments' created successfully.
SQL Server connection closed.


## 2. Connect to SQL SERVER Database and See Available Tables

In real companies, data usually comes from databases, not direct CSV files. Here we connect to a SQLite database and check what tables are available.

This helps understand the database before analysis.

## 3. Read Tables One by One

In [5]:
patients = pd.read_sql('SELECT * FROM patients;', engine)
admissions = pd.read_sql('SELECT * FROM admissions;', engine)
treatments = pd.read_sql('SELECT * FROM treatments;', engine)
billing = pd.read_sql('SELECT * FROM billing_claims;', engine)
pharmacy = pd.read_sql('SELECT * FROM pharmacy_orders;', engine)
labs = pd.read_sql('SELECT * FROM lab_results;', engine)
followups = pd.read_sql('SELECT * FROM followups;', engine)
doctors = pd.read_sql('SELECT * FROM doctors;', engine)
departments = pd.read_sql('SELECT * FROM departments;', engine)
hospitals = pd.read_sql('SELECT * FROM hospitals;', engine)

Tables = ['patients','admissions','treatments','billing','pharmacy','labs','followups','doctors','departments','hospitals']

for table in Tables:
    print(f'{table}:',globals()[table].shape)

patients: (12090, 10)
admissions: (30130, 12)
treatments: (90160, 7)
billing: (30080, 10)
pharmacy: (70090, 8)
labs: (60000, 7)
followups: (30000, 7)
doctors: (260, 7)
departments: (12, 3)
hospitals: (8, 6)


## 4. First-Level Data Understanding

Before cleaning, we need to inspect:

- Number of rows and columns
- Column names
- Sample records
- Data types
- Missing values

This is like a doctor checking symptoms before treatment.

## 5. Create a Simple Data Quality Report

This function checks data quality for one table.

Function logic:

- For each column, find data type
- Count missing values
- Calculate missing percentage
- Count unique values

This is useful because we can apply the same logic to many tables

int ---> "int"
float ---> "float"

In [6]:
def create_quality_report(df, table_name):
    report = pd.DataFrame()
    report['table_name'] = [table_name] * len(df.columns)
    report['column_name'] = df.columns      
    report['data_type'] = df.dtypes.astype(str).values
    report['missing_count'] = df.isna().sum().values
    report['missing_percentage'] = (df.isna().mean() * 100).round(2).values
    report['unique_values'] = df.nunique(dropna=True).values
    return report


quality_reports = []

for table in Tables:
    df = globals()[table]   # access dataframe using table name
    report = create_quality_report(df, table)
    quality_reports.append(report)

quality_report = pd.concat(quality_reports, ignore_index=True)

quality_report.sort_values(
    'missing_percentage',
    ascending=False
).head(20)

,table_name,column_name,data_type,missing_count,missing_percentage,unique_values
6,patients,primary_chronic_condition,object,3249,26.87,7
3,patients,age,float64,211,1.75,101
5,patients,city,object,170,1.41,14
16,admissions,discharge_date,object,251,0.83,1099
18,admissions,diagnosis,object,143,0.47,43
33,billing,gross_amount,float64,120,0.40,29826
26,treatments,treatment_cost,float64,300,0.33,86417
45,pharmacy,line_amount,float64,150,0.21,58529
51,labs,test_value,float64,120,0.20,3559
4,patients,gender,object,0,0.00,10


## 6 Clean Patient Master Data

### Business reason

Patient master data is the foundation. If patient age, gender, or chronic condition is wrong, all later analysis becomes weak.

### Cleaning tasks

    1. Remove duplicate patient records
    2. Convert age into number
    3. Handle impossible ages
    4. Standardize gender labels
    5. Standardize chronic condition labels

In [7]:
patients.head()

,patient_id,patient_code,patient_name,age,gender,city,primary_chronic_condition,insurance_plan,income_segment,registration_date
0,1,P00001,Patient_BKOKF,46.0,Female,Hyderabad,Heart Failure,Premium Insurance,Low,2016-09-25
1,2,P00002,Patient_KXPSW,58.0,Male,Hyderabad,Heart Failure,Basic Insurance,Low,2022-09-12
2,3,P00003,Patient_TJAJT,51.0,Female,Ahmedabad,Hypertension,Government Scheme,Middle,2014-08-02
3,4,P00004,Patient_WYKTO,25.0,Male,None,Heart Failure,Self Pay,Middle,2017-03-23
4,5,P00005,Patient_PFFLA,54.0,Male,None,Asthma,Basic Insurance,Middle,2021-11-22


In [8]:
patients_clean = patients.copy()

print('Before duplicate removal:', patients_clean.shape)
patients_clean = patients_clean.drop_duplicates()
print('After duplicate removal:', patients_clean.shape)

Before duplicate removal: (12090, 10)
After duplicate removal: (12090, 10)


In [9]:
# Convert age to numeric.
# If age contains text or invalid values, errors='coerce' converts them to NaN.
patients_clean['age_clean'] = pd.to_numeric(patients_clean['age'], errors='coerce')

# Mark impossible ages as missing.
patients_clean.loc[patients_clean['age_clean'] < 0, 'age_clean'] = np.nan
patients_clean.loc[patients_clean['age_clean'] > 110, 'age_clean'] = np.nan

# Fill missing age using median because median is less affected by outliers.
median_age = patients_clean['age_clean'].median()
patients_clean['age_clean'] = patients_clean['age_clean'].fillna(median_age)

patients_clean[['age', 'age_clean']].head(20)

,age,age_clean
0,46.0,46.0
1,58.0,58.0
2,51.0,51.0
3,25.0,25.0
4,54.0,54.0
5,NaN,48.0
6,57.0,57.0
7,66.0,66.0
8,62.0,62.0
9,55.0,55.0


In [10]:
# Standardize gender values.
patients_clean['gender_clean'] = patients_clean['gender'].astype(str).str.strip().str.lower()

patients_clean['gender_clean'] = patients_clean['gender_clean'].replace({
    'm': 'Male',
    'male': 'Male',
    'f': 'Female',
    'female': 'Female',
    'femle': 'Female',
    'other': 'Other',
    'unknown': 'Unknown',
    'nan': 'Unknown'
})

patients_clean['gender_clean'].value_counts()

gender_clean
Female     6125
Male       5769
Other       130
Unknown      66
Name: count, dtype: int64

In [11]:
patients_clean['primary_chronic_condition'].value_counts()

primary_chronic_condition
Hypertension     2387
Diabetes         2134
Asthma           1058
COPD              975
Heart Failure     937
CKD               876
Cancer            474
Name: count, dtype: int64

In [12]:
# Standardize chronic condition values.
patients_clean['chronic_condition_clean'] = (
    patients_clean['primary_chronic_condition']
    .astype(str)
    .str.strip()
    .str.title()
)


In [13]:
patients_clean['chronic_condition_clean'].value_counts()

chronic_condition_clean
None             3249
Hypertension     2387
Diabetes         2134
Asthma           1058
Copd              975
Heart Failure     937
Ckd               876
Cancer            474
Name: count, dtype: int64

In [14]:
# Replace missing/blank/unwanted text with Unknown
patients_clean['chronic_condition_clean'] = patients_clean['chronic_condition_clean'].replace({
    'Nan': 'Unknown',
    '': 'Unknown',
    'None': 'Unknown',
    'Null': 'Unknown'
})

In [15]:
patients_clean['chronic_condition_clean'].value_counts()

chronic_condition_clean
Unknown          3249
Hypertension     2387
Diabetes         2134
Asthma           1058
Copd              975
Heart Failure     937
Ckd               876
Cancer            474
Name: count, dtype: int64

In [16]:
# Create chronic flag
# If chronic condition is available, flag = 1
# If missing/unknown, flag = 0
patients_clean['chronic_flag'] = patients_clean['chronic_condition_clean'].apply(
    lambda x: 0 if x == 'Unknown' else 1
)

patients_clean[['primary_chronic_condition', 'chronic_condition_clean', 'chronic_flag']].head()

,primary_chronic_condition,chronic_condition_clean,chronic_flag
0,Heart Failure,Heart Failure,1
1,Heart Failure,Heart Failure,1
2,Hypertension,Hypertension,1
3,Heart Failure,Heart Failure,1
4,Asthma,Asthma,1


In [17]:
patients_clean['chronic_flag'].value_counts()

chronic_flag
1    8841
0    3249
Name: count, dtype: int64

## 7 Clean Admissions Data

### Business reason

Admissions data tells us patient visits, discharge dates, readmissions, department, doctor, and hospital branch.

## Important derived column
    
length_of_stay = discharge_date - admission_date

This is one of the most important healthcare KPIs.

In [18]:
admissions.head()

,admission_id,patient_id,hospital_id,department_id,primary_doctor_id,admission_date,discharge_date,admission_type,diagnosis,severity_level,discharge_status,readmitted_30_days
0,1,3415,8,4,123,2024-03-10,2024-03-16,Emergency,Anemia,Critical,Home,Yes
1,2,3610,8,1,209,2023-05-19,2023-05-20,Planned,Arrhythmia,Moderate,Home,No
2,3,7584,8,3,190,2025-06-16,2025-06-18,Emergency,Joint Replacement,Moderate,Home,Yes
3,4,4459,2,3,176,2025-10-15,2025-10-16,Referral,Joint Replacement,Moderate,Home,No
4,5,5508,3,12,140,2024-05-07,2024-05-23,Planned,Multi Organ Dysfunction,Moderate,Home,No


In [19]:
admissions_clean = admissions.copy()

admissions_clean = admissions_clean.drop_duplicates()

In [20]:
# changing the data type of the date realted columns
admissions_clean['admission_date'] = pd.to_datetime(admissions_clean['admission_date'], errors='coerce')
admissions_clean['discharge_date'] = pd.to_datetime(admissions_clean['discharge_date'], errors='coerce')

In [21]:
admissions_clean['length_of_stay'] = (
    admissions_clean['discharge_date'] - admissions_clean['admission_date']
).dt.days

admissions_clean[['admission_date', 'discharge_date', 'length_of_stay']].head()

,admission_date,discharge_date,length_of_stay
0,2024-03-10,2024-03-16,6.0
1,2023-05-19,2023-05-20,1.0
2,2025-06-16,2025-06-18,2.0
3,2025-10-15,2025-10-16,1.0
4,2024-05-07,2024-05-23,16.0


In [22]:
admissions_clean['discharge_date'].isnull().sum()

np.int64(250)

In [23]:
admissions_clean['admission_date'].isnull().sum()

np.int64(0)

In [24]:
# Handle wrong length of stay values.
# Negative stay means discharge happened before admission, which is impossible.

admissions_clean.loc[admissions_clean['length_of_stay'] < 0, 'length_of_stay'] = np.nan

In [25]:
# Fill missing length of stay using median length of stay.

median_los = admissions_clean['length_of_stay'].median()
admissions_clean['length_of_stay'] = admissions_clean['length_of_stay'].fillna(median_los)

admissions_clean['length_of_stay'].describe()

count    30000.000000
mean         5.915333
std          5.432054
min          1.000000
25%          2.000000
50%          4.000000
75%          8.000000
max         66.000000
Name: length_of_stay, dtype: float64

In [26]:
admissions_clean.head()

,admission_id,patient_id,hospital_id,department_id,primary_doctor_id,admission_date,discharge_date,admission_type,diagnosis,severity_level,discharge_status,readmitted_30_days,length_of_stay
0,1,3415,8,4,123,2024-03-10,2024-03-16,Emergency,Anemia,Critical,Home,Yes,6.0
1,2,3610,8,1,209,2023-05-19,2023-05-20,Planned,Arrhythmia,Moderate,Home,No,1.0
2,3,7584,8,3,190,2025-06-16,2025-06-18,Emergency,Joint Replacement,Moderate,Home,Yes,2.0
3,4,4459,2,3,176,2025-10-15,2025-10-16,Referral,Joint Replacement,Moderate,Home,No,1.0
4,5,5508,3,12,140,2024-05-07,2024-05-23,Planned,Multi Organ Dysfunction,Moderate,Home,No,16.0


In [27]:
admissions_clean['readmitted_30_days'].unique()

array(['Yes', 'No', 'Y', 'FALSE', 'yes', 'TRUE', 'N', 'no'], dtype=object)

In [28]:
# Standardize readmission labels into 1 and 0.
admissions_clean['readmitted_clean'] = admissions_clean['readmitted_30_days'].astype(str).str.strip().str.lower()

admissions_clean['readmitted_flag'] = admissions_clean['readmitted_clean'].replace({
    'yes': 1,
    'y': 1,
    'true': 1,
    'no': 0,
    'n': 0,
    'false': 0,
    'nan': 0
}).astype(int)

admissions_clean['readmitted_flag'].value_counts(normalize=True) * 100

C:\Users\DELL\AppData\Local\Temp\ipykernel_18852\435661268.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  admissions_clean['readmitted_flag'] = admissions_clean['readmitted_clean'].replace({


readmitted_flag
0    51.53
1    48.47
Name: proportion, dtype: float64

## 8. Clean Treatment Cost Data

Treatment cost can contain missing values and extreme outliers.

We will:

    1. Convert cost into numeric
    2. Identify impossible costs
    3. Convert impossible costs to missing
    4. Fill missing cost by treatment-type median
    
Why median by treatment type? Because ICU treatment and basic consultation cannot have the same typical cost.

In [29]:
treatments_clean = treatments.copy()

treatments_clean['treatment_cost'] = pd.to_numeric(
    treatments_clean['treatment_cost'],
    errors='coerce'
)

treatments_clean['cost_outlier_flag'] = 0

In [30]:
treatments_clean.head()

,treatment_id,admission_id,doctor_id,treatment_type,treatment_cost,treatment_date,treatment_status,cost_outlier_flag
0,1,27337,62,Surgery,52636.30,2023-01-01,Completed,0
1,2,982,242,Imaging,7478.12,2023-01-01,Completed,0
2,3,17082,238,Chemotherapy,27451.58,2023-01-01,Completed,0
3,4,4096,1,Lab Package,3379.43,2023-01-01,Pending,0
4,5,26108,227,ICU Care,19993.51,2023-01-01,Completed,0


In [31]:
treatments_clean['cost_outlier_flag'].unique()

array([0])

In [32]:
treatments_clean.loc[
    (treatments_clean['treatment_cost'] < 0) | (treatments_clean['treatment_cost'] > 500000),
    'cost_outlier_flag'
] = 1

print('Number of cost outliers:', treatments_clean['cost_outlier_flag'].sum())

Number of cost outliers: 212


In [33]:
# Convert impossible treatment cost into missing value.

treatments_clean.loc[
    treatments_clean['cost_outlier_flag'] == 1,
    'treatment_cost'
] = np.nan

In [34]:
treatments_clean.head()

,treatment_id,admission_id,doctor_id,treatment_type,treatment_cost,treatment_date,treatment_status,cost_outlier_flag
0,1,27337,62,Surgery,52636.30,2023-01-01,Completed,0
1,2,982,242,Imaging,7478.12,2023-01-01,Completed,0
2,3,17082,238,Chemotherapy,27451.58,2023-01-01,Completed,0
3,4,4096,1,Lab Package,3379.43,2023-01-01,Pending,0
4,5,26108,227,ICU Care,19993.51,2023-01-01,Completed,0


In [35]:
# Fill missing treatment cost using median cost of the same treatment type.
treatments_clean['treatment_cost_clean'] = treatments_clean.groupby('treatment_type')['treatment_cost'].transform(
    lambda values: values.fillna(values.median())
)

treatments_clean[['treatment_type', 'treatment_cost', 'treatment_cost_clean', 'cost_outlier_flag']].head()

,treatment_type,treatment_cost,treatment_cost_clean,cost_outlier_flag
0,Surgery,52636.30,52636.30,0
1,Imaging,7478.12,7478.12,0
2,Chemotherapy,27451.58,27451.58,0
3,Lab Package,3379.43,3379.43,0
4,ICU Care,19993.51,19993.51,0


## 9. Aggregate Transaction Data to Admission Level

One admission can have many treatments, many medicines, and many lab tests. For analysis, we need one row per admission.

So we group by admission_id and calculate totals/counts.

In [36]:
treatment_summary = treatments_clean.groupby('admission_id').agg(
    total_treatment_cost = ('treatment_cost_clean', 'sum'),
    treatment_count = ('treatment_id', 'count'),
    treatment_outlier_count = ('cost_outlier_flag', 'sum')
).reset_index()

treatment_summary.head()

,admission_id,total_treatment_cost,treatment_count,treatment_outlier_count
0,1,13633.72,3,0
1,2,16345.82,2,0
2,3,7095.49,2,0
3,4,7229.08,2,0
4,5,75709.09,3,0


## 10 pharmacy

In [37]:
pharmacy.head()

,order_id,admission_id,patient_id,medicine_name,quantity,unit_price,line_amount,pharmacy_type
0,4573,16572,10603,Salbutamol,12,143.14,1717.63,IP Pharmacy
1,4574,9273,3266,Insulin,10,104.56,1045.57,IP Pharmacy
2,4575,20166,826,Atorvastatin,10,66.94,669.36,IP Pharmacy
3,4576,22020,10512,Chemotherapy Drug,12,52.42,628.99,IP Pharmacy
4,4577,184,2277,Atorvastatin,3,106.36,319.09,IP Pharmacy


In [38]:
pharmacy_clean = pharmacy.copy()
pharmacy_clean['line_amount'] = pd.to_numeric(pharmacy_clean['line_amount'], errors='coerce')
pharmacy_clean['line_amount_clean'] = pharmacy_clean['line_amount'].fillna(pharmacy_clean['line_amount'].median())

pharmacy_summary = pharmacy_clean.groupby('admission_id').agg(
    total_pharmacy_cost = ('line_amount_clean', 'sum'),
    medicine_count = ('order_id', 'count')
).reset_index()

pharmacy_summary.head()

,admission_id,total_pharmacy_cost,medicine_count
0,1,1437.36,2
1,2,1173.55,1
2,3,3801.23,4
3,4,7116.67,5
4,6,3218.53,2


## 11 labs

In [39]:
labs.head()

,lab_id,admission_id,patient_id,test_name,test_value,test_date,result_flag
0,1,2133,11981,CRP,5.99,2023-01-01,Abnormal
1,2,5569,5064,Sodium,2.46,2023-01-01,Normal
2,3,17271,1824,CRP,2.84,2023-01-01,Abnormal
3,4,24486,9939,HbA1c,14.91,2023-01-01,Normal
4,5,21094,8920,Hemoglobin,11.46,2023-01-01,Abnormal


In [40]:
labs_clean = labs.copy()

lab_summary = labs_clean.groupby('admission_id').agg(
    lab_test_count = ('lab_id', 'count'),
    abnormal_lab_count = ('result_flag', lambda x: x.isin(['Abnormal', 'Critical']).sum()),
    critical_lab_count = ('result_flag', lambda x: (x == 'Critical').sum())
).reset_index()

lab_summary.head()

,admission_id,lab_test_count,abnormal_lab_count,critical_lab_count
0,1,1,1,0
1,2,1,0,0
2,3,2,1,0
3,4,1,0,0
4,5,1,0,0


## 12 billing

In [41]:
billing_clean = billing.copy()

In [42]:
billing_clean.dtypes

bill_id              int64
admission_id         int64
patient_id           int64
payer_type          object
gross_amount       float64
discount_amount    float64
insurance_paid     float64
patient_paid       float64
claim_status        object
bill_date           object
dtype: object

In [43]:
billing_clean.isnull().sum()

bill_id              0
admission_id         0
patient_id           0
payer_type           0
gross_amount       120
discount_amount      0
insurance_paid       0
patient_paid         0
claim_status         0
bill_date            0
dtype: int64

In [44]:
# Standardize claim_status values.
billing_clean['claim_status'] = (
    billing_clean['claim_status']
    .astype(str)
    .str.strip()
    .str.title()
)

billing_clean['claim_status'] = billing_clean['claim_status'].replace({'Under Review':'Under_Review'})

In [45]:
# Converting bill_date to date_time

billing_clean['bill_date'] = pd.to_datetime(billing_clean['bill_date'], errors='coerce')

In [46]:
# Filling missing values for gross_amount

billing_clean['gross_amount'] = billing_clean['gross_amount'].fillna(
    billing_clean['insurance_paid'] +
    billing_clean['patient_paid'] +
    billing_clean['discount_amount']
)

In [47]:
billing_clean['gross_amount'].isnull().sum()

np.int64(0)

In [48]:
billing_clean.dtypes

bill_id                     int64
admission_id                int64
patient_id                  int64
payer_type                 object
gross_amount              float64
discount_amount           float64
insurance_paid            float64
patient_paid              float64
claim_status               object
bill_date          datetime64[ns]
dtype: object

## 13 followups

In [49]:
followups_clean = followups.copy()

In [50]:
followups_clean.isnull().sum()

followup_id           0
admission_id          0
patient_id            0
followup_due_date     0
followup_completed    0
followup_mode         0
followup_outcome      0
dtype: int64

In [51]:
followups_clean.dtypes

followup_id            int64
admission_id           int64
patient_id             int64
followup_due_date     object
followup_completed    object
followup_mode         object
followup_outcome      object
dtype: object

In [52]:
followups_clean['followup_due_date'] = pd.to_datetime(followups_clean['followup_due_date'], errors='coerce')

In [53]:
followups_clean.dtypes

followup_id                    int64
admission_id                   int64
patient_id                     int64
followup_due_date     datetime64[ns]
followup_completed            object
followup_mode                 object
followup_outcome              object
dtype: object

# 14 doctors

In [54]:
doctors_clean = doctors.copy()

In [55]:
doctors_clean

,doctor_id,doctor_name,department_id,specialization,hospital_id,experience_years,doctor_grade
0,1,Dr. Sai Sharma,4,General Medicine,8,9,Consultant
1,2,Dr. Raj Nair,12,ICU,7,10,Resident
2,3,Dr. Ishaan Khan,9,Emergency,6,16,Consultant
3,4,Dr. Sai Rao,7,Gastroenterology,7,7,Consultant
4,5,Dr. Priya Menon,2,Neurology,2,12,Senior Consultant
...,...,...,...,...,...,...,...
255,256,Dr. Raj Das,7,Gastroenterology,5,7,Senior Consultant
256,257,Dr. Anaya Rao,12,ICU,1,11,Consultant
257,258,Dr. Arjun Khan,9,Emergency,8,20,Senior Consultant
258,259,Dr. Ishaan Naidu,7,Gastroenterology,8,14,Consultant


In [56]:
doctors_clean.isnull().sum()

doctor_id           0
doctor_name         0
department_id       0
specialization      0
hospital_id         0
experience_years    0
doctor_grade        0
dtype: int64

In [57]:
doctors_clean['doctor_name'].value_counts()

doctor_name
Dr. Aditya Nair       4
Dr. Reyansh Chopra    4
Dr. Neha Rao          4
Dr. Sai Rao           3
Dr. Aditya Reddy      3
                     ..
Dr. Reyansh Jain      1
Dr. Sara Verma        1
Dr. Sara Rao          1
Dr. Raj Das           1
Dr. Arjun Khan        1
Name: count, Length: 188, dtype: int64

In [58]:
doctors_clean['doctor_grade'].unique()

array(['Consultant', 'Resident', 'Senior Consultant',
       'Visiting Consultant'], dtype=object)

## 15 departments

In [59]:
departments_clean = departments.copy()

In [60]:
departments_clean

,department_id,department_name,care_complexity
0,1,Cardiology,High
1,2,Neurology,High
2,3,Orthopedics,Medium
3,4,General Medicine,Medium
4,5,Pulmonology,High
5,6,Nephrology,High
6,7,Gastroenterology,Medium
7,8,Oncology,Very High
8,9,Emergency,High
9,10,Endocrinology,Medium


In [61]:
departments_clean.shape

(12, 3)

In [62]:
departments_clean.isnull().sum()

department_id      0
department_name    0
care_complexity    0
dtype: int64

## 16 hospitals

In [63]:
hospitals_clean = hospitals

In [64]:
hospitals_clean

,hospital_id,hospital_name,city,ownership_type,bed_capacity,launch_year
0,1,MedNova Hospital - Hyderabad,Hyderabad,Private,420,2009
1,2,MedNova Hospital - Bengaluru,Bengaluru,Private,360,2012
2,3,MedNova Hospital - Chennai,Chennai,Trust,310,2014
3,4,MedNova Hospital - Pune,Pune,Private,280,2018
4,5,MedNova Hospital - Mumbai,Mumbai,Private,390,2011
5,6,MedNova Hospital - Delhi,Delhi,Trust,330,2015
6,7,MedNova Hospital - Vijayawada,Vijayawada,Private,240,2020
7,8,MedNova Hospital - Warangal,Warangal,Private,210,2021


In [65]:
hospitals_clean.isnull().sum()

hospital_id       0
hospital_name     0
city              0
ownership_type    0
bed_capacity      0
launch_year       0
dtype: int64

In [73]:
Tabels = ['patients_clean','admissions_clean','treatments_clean','pharmacy_clean','labs_clean','billing_clean','followups_clean','doctors_clean','departments_clean','hospitals_clean']

for i in Tabels:
    df = globals()[i]
    print(df.shape)

(12090, 14)
(30000, 15)
(90160, 9)
(70090, 9)
(60000, 7)
(30080, 10)
(30000, 7)
(260, 7)
(12, 3)
(8, 6)


In [76]:
Tabels = ['patients_clean','admissions_clean','treatments_clean','pharmacy_clean','labs_clean','billing_clean','followups_clean','doctors_clean','departments_clean','hospitals_clean']

folder_path = "cleaned_data_files"

os.makedirs(folder_path, exist_ok=True)

for i in Tabels:
    df = globals()[i]
    df.to_csv(f"{folder_path}/{i}.csv", index=False)

print("All tables saved as CSV files.")

All tables saved as CSV files.
